In [21]:
from pathlib import Path
import re
import socket

import pandas as pd
import pyfame as pf
from pyfame.pyfame import BoEFAMEReader

# =========================================================
# 1. CONFIG
# =========================================================

PROJECT_ROOT = Path(r"C:\Users\344792\Gokce\GIT PROJECTS\DisaggCPI\CPI-disaggregation-in-PT")
OUTPUT_DIR = PROJECT_ROOT / "data" / "fame_exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VINTAGE_START = "M15"
VINTAGE_END = "M26"
FREQUENCY = "quarterly"
END_OF_PERIOD_DATES = True
EXPORT_TO_CSV_FROM_PYFAME = False

FAME_LOCAL_HOST = "localhost"
FAME_LOCAL_PORT = 12345

reader = BoEFAMEReader()

# =========================================================
# 2. ESTIMATION SERIES MAP
# =========================================================

# Logical names used to construct the final estimation_data table
ESTIMATION_SERIES_MAP = {
    "date": "date",
    "cpi": "stifidx",
    "core_gds": "stif_gds_xeatfnab",
    "services": "stif_srv",
    "food_at": "food_at_expr",
    "energy": "stif_pu",
    "pmdef": "pmdef",
    "eer": "eer",
    "infl_exp": "infl_exp_expr",
}

REQUIRED_ESTIMATION_COLUMNS = [
    "date", "cpi", "core_gds", "services", "food_at", "energy", "pmdef", "eer", "infl_exp"
]

# =========================================================
# 3. EXPRESSION OVERRIDES
# =========================================================

# These mirror the Excel estimation sheet logic:
# e.g. B10 contains the FAME expression, and B12 is =@FAMEData(B10,...)

INFL_EXP_FAME_EXPR = 'BPROJA(conv2qa(cgrp.m),PUB12MEDIAN.Q,"E")'

EXPRESSION_OVERRIDES = {
    "stifidx": "conv2qa(X12(stifidx.m,MULT,1990m2,2025m12))",
    "stif_srv": "conv2qa(X12(stif_srv.m,MULT,1990m2,2025m12))",
    "stif_gds_xeatfnab": "conv2qa(X12(stif_gds_xeatfnab.m,MULT,1990m2,2025m12))",
    "food_at_expr": "conv2qa(X12(cpi_aggregation(stif_fnab.m, booze.m, fags.m,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#),MULT,1990m2,2025m12))",
    "stif_pu": "conv2qa(X12(stif_pu.m,MULT,1990m2,2025m12))",
    "infl_exp_expr": INFL_EXP_FAME_EXPR,
}

ALL_CODES = sorted(set(v for k, v in ESTIMATION_SERIES_MAP.items() if k != "date"))

# =========================================================
# 4. HELPERS
# =========================================================

_VINTAGE_LETTERS = ["F", "M", "A", "N"]


def is_port_open(host: str, port: int, timeout: float = 2.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        return s.connect_ex((host, port)) == 0


def ensure_fame_service_available(host: str = FAME_LOCAL_HOST, port: int = FAME_LOCAL_PORT):
    if not is_port_open(host, port):
        raise ConnectionError(
            f"FAME local service is not reachable at {host}:{port}. "
            "Please start the local FAME web service/tunnel."
        )


def _round_to_index(label: str) -> int:
    m = re.match(r"^([FMAN])(\d{2})$", str(label).strip().upper())
    if not m:
        raise ValueError(f"Bad round label: {label!r}")
    yy = int(m.group(2))
    return yy * 4 + _VINTAGE_LETTERS.index(m.group(1))


def _vintage_sort_key(name: str):
    text = str(name).strip().upper()
    m = re.fullmatch(r"([FMAN])(\d{2})", text)
    if m:
        return (1, _round_to_index(text))
    match = re.search(r"(\d+)", text)
    return (0, int(match.group(1))) if match else (-1, -1)


def latest_vintage_column(df: pd.DataFrame) -> str:
    candidates = [c for c in df.columns if c != "date"]
    if not candidates:
        raise ValueError("No vintage columns found. Expected at least one column besides 'date'.")
    return sorted(candidates, key=_vintage_sort_key)[-1]


def latest_series(df: pd.DataFrame, output_name: str) -> pd.DataFrame:
    col = latest_vintage_column(df)
    return df[["date", col]].rename(columns={col: output_name}).copy()


def normalize_quarter_end_dates(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    out = df.copy()
    raw = out[date_col]
    numeric = pd.to_numeric(raw, errors="coerce")

    if numeric.notna().any() and pd.api.types.is_numeric_dtype(raw):
        dt = pd.to_datetime(numeric, unit="ms", errors="coerce")
    else:
        dt = pd.to_datetime(raw, errors="coerce")
        if dt.notna().any() and dt.dropna().dt.year.median() < 1980 and numeric.notna().any():
            dt_ms = pd.to_datetime(numeric, unit="ms", errors="coerce")
            if dt_ms.notna().sum() >= dt.notna().sum():
                dt = dt_ms

    quarter_start = dt.dt.is_quarter_start.fillna(False)
    dt = dt.where(~quarter_start, dt - pd.Timedelta(days=1))
    out[date_col] = dt.dt.to_period("Q").dt.to_timestamp(how="end").dt.normalize()
    return out


def reorder_and_validate(df: pd.DataFrame, required_columns: list, dataset_name: str) -> pd.DataFrame:
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{dataset_name} missing columns: {missing}")
    return df[required_columns].sort_values("date").reset_index(drop=True)


def read_expression_to_latest(expr: str) -> pd.DataFrame:
    """
    Read a FAME expression and coerce it to:
      date + latest-value-column
    """
    df = reader.read(expr)
    if df is None or df.empty:
        raise ValueError(f"FAME returned empty data for expression: {expr}")

    df = df.copy()

    if "index" in df.columns and "value" in df.columns:
        df = df.rename(columns={"index": "date", "value": VINTAGE_END})
    elif "date" in df.columns and "value" in df.columns:
        df = df.rename(columns={"value": VINTAGE_END})
    else:
        if df.shape[1] < 2:
            raise ValueError(f"Unexpected shape from expression read: {expr}")
        df = df.iloc[:, :2].copy()
        df.columns = ["date", VINTAGE_END]

    df = normalize_quarter_end_dates(df)
    df = df.dropna(subset=["date", VINTAGE_END]).sort_values("date").reset_index(drop=True)
    return df


def download_plain_variable(var_name: str) -> pd.DataFrame:
    result = pf.download_vintaged_fame_data(
        variables=[var_name],
        vintage_start=VINTAGE_START,
        vintage_end=VINTAGE_END,
        frequency=FREQUENCY,
        end_of_period_dates=END_OF_PERIOD_DATES,
        export_to_csv=EXPORT_TO_CSV_FROM_PYFAME,
    )

    if not isinstance(result, dict) or var_name not in result:
        raise KeyError(f"Missing series from FAME response: {var_name}")

    df = result[var_name]
    if df is None or df.empty:
        raise ValueError(f"FAME returned empty data for variable: {var_name}")

    df = normalize_quarter_end_dates(df)
    return df


def build_raw_data(all_codes: list) -> dict:
    raw_data = {}

    print("\nReading estimation data...")
    for code in all_codes:
        print(f"  -> {code}")

        if code in EXPRESSION_OVERRIDES:
            df = read_expression_to_latest(EXPRESSION_OVERRIDES[code])
        else:
            df = download_plain_variable(code)

        raw_data[code] = df

    return raw_data


def build_panel(raw_data: dict, mapping: dict) -> pd.DataFrame:
    panel = None
    for output_name, fame_code in mapping.items():
        if output_name == "date":
            continue
        if fame_code not in raw_data:
            raise KeyError(f"Missing FAME code in downloaded data: {fame_code}")
        current = latest_series(raw_data[fame_code], output_name)
        panel = current if panel is None else panel.merge(current, on="date", how="outer")
    return panel.sort_values("date").reset_index(drop=True)

# =========================================================
# 5. MAIN
# =========================================================

ensure_fame_service_available()

print("Setup complete.")
print(f"Output folder: {OUTPUT_DIR}")
print(f"FAME endpoint expected by pyfame: {FAME_LOCAL_HOST}:{FAME_LOCAL_PORT}")

raw_data = build_raw_data(ALL_CODES)

estimation_data = build_panel(raw_data, ESTIMATION_SERIES_MAP)

estimation_data = estimation_data[
    (estimation_data["date"] >= pd.Timestamp("1990-04-01")) &
    (estimation_data["date"] <= pd.Timestamp("2025-12-31"))
].copy()

estimation_data = reorder_and_validate(
    estimation_data, REQUIRED_ESTIMATION_COLUMNS, "Estimation_data"
)

print("\nBuilt estimation dataset.")
print(estimation_data.head())
print(estimation_data.tail())

estimation_data.to_parquet(OUTPUT_DIR / "estimation_data_latest.parquet", index=False)
estimation_data.to_csv(OUTPUT_DIR / "estimation_data_latest.csv", index=False)

print("\nDone.")
print(f"Wrote: {OUTPUT_DIR / 'estimation_data_latest.parquet'}")
print(f"Wrote: {OUTPUT_DIR / 'estimation_data_latest.csv'}")


Setup complete.
Output folder: C:\Users\344792\Gokce\GIT PROJECTS\DisaggCPI\CPI-disaggregation-in-PT\data\fame_exports
FAME endpoint expected by pyfame: localhost:12345

Reading estimation data...
  -> eer


100%|██████████| 45/45 [00:15<00:00,  2.91it/s]


  -> food_at_expr
  -> infl_exp_expr
  -> pmdef


100%|██████████| 45/45 [00:09<00:00,  4.80it/s]


  -> stif_gds_xeatfnab
  -> stif_pu
  -> stif_srv
  -> stifidx

Built estimation dataset.
        date       cpi  core_gds  services    food_at    energy     pmdef  \
0 1990-06-30  0.559306  0.559306  0.559306  99.445968  0.559306  0.577736   
1 1990-09-30  0.566150  0.566150  0.566150  99.670858  0.566150  0.559306   
2 1990-12-31  0.562934  0.562934  0.562934  99.118131  0.562934  0.566150   
3 1991-03-31  0.572353  0.572353  0.572353  96.421142  0.572353  0.562934   
4 1991-06-30  0.583560  0.583560  0.583560  95.722653  0.583560  0.572353   

         eer  infl_exp  
0  93.068646  9.103957  
1  99.445968  7.479474  
2  99.670858  7.172456  
3  99.118131  5.585646  
4  96.421142  4.608519  
          date       cpi  core_gds  services    food_at    energy     pmdef  \
138 2024-12-31  0.998951  0.998951  0.998951  84.062595  0.998951  0.976247   
139 2025-03-31  0.992517  0.992517  0.992517  85.431326  0.992517  0.998951   
140 2025-06-30  0.986093  0.986093  0.986093  85.000674  0.9